# CRC LoRA — Comprehensive Analysis

Each figure is tagged for its intended use:
- **[MAIN]** — candidate for paper main figure
- **[SUPP]** — candidate for supplementary figure
- **[EXPL]** — exploratory / sanity check, unlikely to publish

Figures are saved as both `.pdf` (for paper) and `.png` (for quick view) under `figures/crc/`.


In [ ]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from matplotlib.gridspec import GridSpec
import seaborn as sns
from scipy import stats

warnings.filterwarnings('ignore', category=FutureWarning)

# --- Style: Nature-ish, open axes, frameless legends, editable text in PDF ---
sns.set_theme(style="white", context="paper", font_scale=1.0)
mpl.rcParams.update({
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'axes.titleweight':  'bold',
    'axes.titlesize':    10,
    'axes.labelsize':    9,
    'axes.titlelocation': 'left',     # lowercase bold left-aligned panel labels (Nature style)
    'legend.frameon':    False,
    'legend.fontsize':   8,
    'xtick.labelsize':   8,
    'ytick.labelsize':   8,
    'pdf.fonttype':      42,
    'ps.fonttype':       42,
    'figure.dpi':        110,
    'savefig.dpi':       300,
})

PROJECT  = Path('/hpc/group/jilab/tc459/MorphPT')
DATASET  = 'crc'
SEED     = 42
EXP_DIR  = PROJECT / 'experiments' / 'lora_probing_crc_multi_10.0x_mse'   # update if rerun with new tiles
CACHE    = PROJECT / 'cache_crc'

SAVE_FIGS = True
FIG_DIR   = PROJECT / 'figures' / 'crc'
if SAVE_FIGS:
    FIG_DIR.mkdir(parents=True, exist_ok=True)

# Optional: path to frozen-baseline predictions. Set to None if not available.
FROZEN_DIR = None    # e.g. PROJECT / 'experiments' / 'frozen_mlp_crc_top400'

# Compartment palette — used everywhere
COMPARTMENT_COLORS = {
    'epithelial': '#2ca58d',  # teal
    'stromal':    '#c66f1f',  # warm orange/brown
    'lymphoid':   '#7e57c2',  # purple (B + T + plasma)
    'myeloid':    '#e55c5c',  # red
    'endothelial':'#3a86ff',  # blue
    'smooth_muscle':'#8b5e34',# brown
    'cycling':    '#ffb703',  # yellow
    'other':      '#9aa5b1',  # gray
}

print('Setup complete.')


## 1. Load data and compute derived quantities

In [ ]:
# 1) Predictions on test set
y_true = np.load(EXP_DIR / f'test_y_true_seed_{SEED}.npy')   # (N_test, G)
y_pred = np.load(EXP_DIR / f'test_y_pred_seed_{SEED}.npy')

# 2) Per-gene results
df_res = pd.read_csv(EXP_DIR / 'multi_lora_hybrid_results.csv')

# 3) Gene stats (variance, coverage on training split)
gene_stats = pd.read_csv(CACHE / 'per_gene' / f'top400_variance_mincov0.1_train_seed{SEED}.csv')

# 4) Merge
df = pd.merge(df_res, gene_stats, on=['gene_idx', 'gene_name'], how='inner')

# Map gene_idx → column position in y_true / y_pred
gene_idx_to_col = {g: i for i, g in enumerate(df_res['gene_idx'].values)}
df['col_idx'] = df['gene_idx'].map(gene_idx_to_col)

# Add per-gene true/pred mean and std (test set)
df['true_mean_z'] = y_true.mean(axis=0)[df['col_idx']]
df['pred_mean_z'] = y_pred.mean(axis=0)[df['col_idx']]
df['true_std_z']  = y_true.std (axis=0)[df['col_idx']]
df['pred_std_z']  = y_pred.std (axis=0)[df['col_idx']]
df['shrinkage']   = df['pred_std_z'] / df['true_std_z']

# Sort by r descending — this becomes our default order
df = df.sort_values('test_pearson_s42', ascending=False).reset_index(drop=True)

# 5) Meta + splits — needed for spatial plots & per-tile decomposition
meta = pd.read_csv(CACHE / 'meta.csv')
all_meta_parts = [meta[meta['split'] == s].copy() for s in ['train', 'val', 'test']]
all_meta = pd.concat(all_meta_parts, axis=0).reset_index(drop=True)

splits = np.load(EXP_DIR / f'splits_seed_{SEED}.npz')
test_idx = splits['test_idx']
test_meta = all_meta.iloc[test_idx].reset_index(drop=True)
TEST_TILES = sorted([int(t) for t in splits['test_tiles']])

assert len(test_meta) == y_true.shape[0], \
    f"meta cells ({len(test_meta)}) != y_true rows ({y_true.shape[0]})"

print(f'Test cells   : {y_true.shape[0]:,}')
print(f'Genes        : {y_true.shape[1]}')
print(f'Test tiles   : {TEST_TILES}')
print(f'Mean test r  : {df["test_pearson_s42"].mean():.4f}')
print(f'Median test r: {df["test_pearson_s42"].median():.4f}')


### Compartment annotation (gene-level only)

Each of the 400 panel genes is mapped to a compartment via a curated marker dictionary
(epithelial, stromal, lymphoid, myeloid, endothelial, smooth_muscle, cycling, other).
Genes not in the dictionary go to `other`.

> **Note**: Cell-level compartment inference (e.g. "this cell is epithelial because PIGR is
> high") is intentionally **not** done here — that requires either ground-truth cell type
> annotation or a separately-validated proxy. Once cell types are available, the
> compartment-stratified scatter (canonical "magnitude vs binary detection" diagnostic)
> can be added back.

In [ ]:
# Curated CRC marker dictionary. Add/remove as you see fit.
# Aim is *purity*, not coverage — only assign genes that are clearly compartment-specific.
COMPARTMENT_MARKERS = {
    'epithelial': [
        # Pan-epithelial / colon-specific
        'EPCAM', 'KRT8', 'KRT18', 'KRT19', 'KRT20', 'CDX2', 'VIL1',
        'PIGR', 'PHGR1', 'MUC1', 'MUC2', 'MUC5AC', 'MUC12', 'MUC13',
        'TFF1', 'TFF3', 'CEACAM1', 'CEACAM5', 'CEACAM6', 'CEACAM7',
        'FABP1', 'GUCA2A', 'GUCA2B', 'AQP8', 'CA1', 'CA2', 'CA4',
        'SLC26A3', 'CLCA1', 'ZG16', 'REG4', 'KRT5', 'KRT15', 'CLDN3',
        'CLDN4', 'CLDN7', 'OCLN', 'LGR5', 'OLFM4', 'ASCL2',
    ],
    'stromal': [
        # Fibroblast / ECM / mesenchymal
        'COL1A1', 'COL1A2', 'COL3A1', 'COL4A1', 'COL4A2', 'COL5A1', 'COL5A2',
        'COL6A1', 'COL6A2', 'COL6A3', 'COL11A1', 'COL12A1',
        'FAP', 'PDGFRA', 'PDGFRB', 'THY1', 'DCN', 'LUM', 'BGN',
        'SPARC', 'POSTN', 'FN1', 'MMP2', 'MMP11', 'MMP14',
        'IGFBP7', 'IGFBP3', 'LGALS1', 'VIM', 'S100A4',
        'TAGLN', 'CALD1', 'ACTA2',
    ],
    'lymphoid': [
        # Plasma cells, B cells, T cells, NK
        'IGKC', 'IGHG1', 'IGHG2', 'IGHG3', 'IGHG4', 'IGHA1', 'IGHA2', 'IGHM', 'IGHD',
        'IGLC1', 'IGLC2', 'IGLC3', 'JCHAIN', 'MZB1', 'XBP1',
        'CD79A', 'CD79B', 'MS4A1', 'CD19', 'CD22', 'CD20',
        'CD3D', 'CD3E', 'CD3G', 'CD4', 'CD8A', 'CD8B',
        'TRAC', 'TRBC1', 'TRBC2', 'CCL5', 'GZMA', 'GZMB', 'GZMK',
        'NKG7', 'KLRD1', 'KLRB1', 'PRF1', 'IL7R', 'FOXP3', 'CTLA4',
    ],
    'myeloid': [
        'CD68', 'CD163', 'CSF1R', 'MARCO', 'AIF1', 'C1QA', 'C1QB', 'C1QC',
        'ITGAX', 'ITGAM', 'CD14', 'FCGR3A', 'FCER1G',
        'S100A8', 'S100A9', 'LYZ', 'TYROBP', 'APOE', 'MRC1',
    ],
    'endothelial': [
        'PECAM1', 'VWF', 'CDH5', 'CLDN5', 'KDR', 'ENG', 'TEK',
        'PLVAP', 'EGFL7', 'ESAM', 'ICAM2',
    ],
    'smooth_muscle': [
        'ACTG2', 'MYH11', 'CNN1', 'DES', 'MYL9', 'MYLK',
    ],
    'cycling': [
        'MKI67', 'TOP2A', 'PCNA', 'AURKB', 'BIRC5', 'CDK1', 'CCNB1',
        'CCNB2', 'CCNA2', 'UBE2C', 'TYMS',
    ],
}

# Build gene → compartment map
gene_to_comp = {}
for comp, markers in COMPARTMENT_MARKERS.items():
    for g in markers:
        gene_to_comp[g] = comp

df['compartment'] = df['gene_name'].map(gene_to_comp).fillna('other')

# Report panel coverage
panel_genes = set(df['gene_name'])
print('Panel coverage of marker dictionary:')
for comp, markers in COMPARTMENT_MARKERS.items():
    in_panel = [m for m in markers if m in panel_genes]
    print(f'  {comp:14s}: {len(in_panel):>3d} / {len(markers):>3d}  '
          f'(e.g. {", ".join(in_panel[:6])}{"..." if len(in_panel) > 6 else ""})')

print('\nCompartment distribution among 400 panel genes:')
print(df['compartment'].value_counts().to_string())


## Figure 1 — per-gene predictability overview  [MAIN candidate]

Four panels:
- **a** distribution of test Pearson r across all 400 genes
- **b** top-N mean r curves (by R, by training variance)
- **c** r vs gene coverage (does the model just predict highly-expressed genes?)
- **d** r vs gene variance (does the model just predict highly-variable genes?)

This is the "shape of the result" figure — gives the reviewer an immediate sense of *how broad* the signal is.


In [ ]:
fig = plt.figure(figsize=(11, 7.5))
gs = GridSpec(2, 2, figure=fig, hspace=0.42, wspace=0.30)

# ---- panel a: histogram of test r ----
ax = fig.add_subplot(gs[0, 0])
ax.hist(df['test_pearson_s42'], bins=40, color='#4A90D9',
        edgecolor='white', linewidth=0.4, alpha=0.85)
for q in [0.25, 0.5, 0.75, 0.95]:
    qv = df['test_pearson_s42'].quantile(q)
    ax.axvline(qv, color='black', ls=':', lw=0.8, alpha=0.6)
    ax.text(qv, ax.get_ylim()[1] * 0.96, f' q{int(q*100)}={qv:.2f}',
            fontsize=7, ha='left', va='top', rotation=0, alpha=0.8)
ax.axvline(0, color='red', ls='--', lw=0.8, alpha=0.6)
ax.set_xlabel('test Pearson r')
ax.set_ylabel('# genes')
ax.set_title('a   per-gene r distribution')
ax.text(0.97, 0.97,
        f"n = {len(df)}\nmean = {df['test_pearson_s42'].mean():.3f}\n"
        f"median = {df['test_pearson_s42'].median():.3f}\n"
        f"r > 0.5: {(df['test_pearson_s42'] > 0.5).sum()} genes\n"
        f"r > 0.3: {(df['test_pearson_s42'] > 0.3).sum()} genes",
        transform=ax.transAxes, va='top', ha='right',
        fontsize=8, family='monospace',
        bbox=dict(boxstyle='round,pad=0.4', facecolor='white',
                  edgecolor='#cccccc', alpha=0.95))

# ---- panel b: top-N curves ----
ax = fig.add_subplot(gs[0, 1])
df_byr   = df.sort_values('test_pearson_s42', ascending=False).reset_index(drop=True)
df_byvar = df.sort_values('variance', ascending=False).reset_index(drop=True)
ks = np.arange(1, len(df) + 1)
mean_byr   = df_byr['test_pearson_s42'].expanding().mean()
mean_byvar = df_byvar['test_pearson_s42'].expanding().mean()

ax.plot(ks, mean_byr,   color='#1f77b4', lw=1.6, label='top-N by test r')
ax.plot(ks, mean_byvar, color='#9aa5b1', lw=1.6, label='top-N by train variance')
for k in [10, 50, 200]:
    ax.axvline(k, color='black', ls=':', lw=0.5, alpha=0.4)
ax.set_xscale('log')
ax.set_xlabel('N (top genes)')
ax.set_ylabel('mean test r over top-N')
ax.set_title('b   top-N predictability curves')
ax.legend(loc='upper right')

# ---- panel c: r vs coverage ----
ax = fig.add_subplot(gs[1, 0])
for comp, sub in df.groupby('compartment'):
    ax.scatter(sub['coverage'], sub['test_pearson_s42'],
               c=COMPARTMENT_COLORS.get(comp, '#9aa5b1'),
               s=14, alpha=0.7, edgecolors='none',
               label=f'{comp} (n={len(sub)})')
# Spearman for monotonic association
rho_cov, p_cov = stats.spearmanr(df['coverage'], df['test_pearson_s42'])
ax.text(0.04, 0.97, f'Spearman ρ = {rho_cov:.2f}',
        transform=ax.transAxes, va='top', fontsize=8, family='monospace',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.9, edgecolor='#cccccc'))
ax.set_xscale('log')
ax.set_xlabel('train coverage (fraction of cells expressing)')
ax.set_ylabel('test Pearson r')
ax.set_title('c   r vs coverage')
ax.legend(loc='lower right', ncol=1, fontsize=6.5, markerscale=1.2)

# ---- panel d: r vs variance ----
ax = fig.add_subplot(gs[1, 1])
for comp, sub in df.groupby('compartment'):
    ax.scatter(sub['variance'], sub['test_pearson_s42'],
               c=COMPARTMENT_COLORS.get(comp, '#9aa5b1'),
               s=14, alpha=0.7, edgecolors='none')
rho_var, p_var = stats.spearmanr(df['variance'], df['test_pearson_s42'])
ax.text(0.04, 0.97, f'Spearman ρ = {rho_var:.2f}',
        transform=ax.transAxes, va='top', fontsize=8, family='monospace',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.9, edgecolor='#cccccc'))
ax.set_xscale('log')
ax.set_xlabel('train variance (raw expression space)')
ax.set_ylabel('test Pearson r')
ax.set_title('d   r vs variance')

if SAVE_FIGS:
    fig.savefig(FIG_DIR / 'fig1_overview.pdf', bbox_inches='tight')
    fig.savefig(FIG_DIR / 'fig1_overview.png', bbox_inches='tight')
plt.show()


## Figure 2 — top predictable genes are CRC compartment markers  [MAIN candidate]

This is the **biological interpretation** figure — answers "why does the model work?"

- **a** top-30 genes ranked by test r, colored by compartment; bar = mean r ± seed std (if multi-seed)
- **b** compartment composition by r quartile — predicted genes are enriched for known CRC compartment markers


In [ ]:
fig = plt.figure(figsize=(11, 7))
gs = GridSpec(1, 2, figure=fig, width_ratios=[1.4, 1.0], wspace=0.35)

# ---- panel a: top-30 horizontal bar ----
ax = fig.add_subplot(gs[0, 0])
top_n = 30
top = df.head(top_n).iloc[::-1]    # reverse for highest at top
colors = [COMPARTMENT_COLORS.get(c, '#9aa5b1') for c in top['compartment']]

bars = ax.barh(np.arange(len(top)), top['test_pearson_s42'],
               color=colors, edgecolor='none', height=0.78)
# Optional: add std error bars if test_pearson_std exists (multi-seed)
if 'test_pearson_std' in top.columns and top['test_pearson_std'].notna().any():
    ax.errorbar(top['test_pearson_s42'], np.arange(len(top)),
                xerr=top['test_pearson_std'], fmt='none',
                ecolor='black', elinewidth=0.6, capsize=2)

ax.set_yticks(np.arange(len(top)))
ax.set_yticklabels(top['gene_name'], fontsize=8)
ax.set_xlabel('test Pearson r')
ax.set_title(f'a   top {top_n} genes by test r (CRC)')
ax.set_xlim(0, max(1.0, top['test_pearson_s42'].max() * 1.05))
ax.axvline(df['test_pearson_s42'].mean(), color='black', ls=':', lw=0.7,
           label=f"all-gene mean ({df['test_pearson_s42'].mean():.2f})")
ax.legend(loc='lower right', fontsize=7)

# Compartment legend (custom, since bars share color by compartment)
present_comps = top['compartment'].unique().tolist()
legend_handles = [mpl.patches.Patch(color=COMPARTMENT_COLORS.get(c, '#9aa5b1'),
                                     label=c) for c in present_comps]
ax.legend(handles=legend_handles, loc='lower right',
          fontsize=7, ncol=1, title='compartment', title_fontsize=7)

# ---- panel b: quartile compartment composition ----
ax = fig.add_subplot(gs[0, 1])
df_q = df.copy()
df_q['quartile'] = pd.qcut(df_q['test_pearson_s42'], 4,
                            labels=['Q1\n(lowest r)', 'Q2', 'Q3', 'Q4\n(highest r)'])

# Order compartments by total presence in highest quartile, then alphabetically
comp_order = (df_q[df_q['quartile'] == 'Q4\n(highest r)']['compartment']
              .value_counts().index.tolist())
remaining = [c for c in df_q['compartment'].unique() if c not in comp_order]
comp_order = comp_order + sorted(remaining)

table = (df_q.groupby('quartile')['compartment']
         .value_counts(normalize=True).unstack(fill_value=0))
table = table[[c for c in comp_order if c in table.columns]]

bottom = np.zeros(len(table))
for c in table.columns:
    ax.bar(np.arange(len(table)), table[c].values, bottom=bottom,
           color=COMPARTMENT_COLORS.get(c, '#9aa5b1'),
           edgecolor='white', linewidth=0.6, label=c)
    bottom = bottom + table[c].values

ax.set_xticks(np.arange(len(table)))
ax.set_xticklabels(table.index)
ax.set_ylabel('fraction of genes')
ax.set_ylim(0, 1)
ax.set_title('b   compartment composition by r quartile')
ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=7)

if SAVE_FIGS:
    fig.savefig(FIG_DIR / 'fig2_top_genes_compartments.pdf', bbox_inches='tight')
    fig.savefig(FIG_DIR / 'fig2_top_genes_compartments.png', bbox_inches='tight')
plt.show()

print('Top-30 gene composition:')
print(top.iloc[::-1][['gene_name', 'test_pearson_s42', 'compartment',
                      'coverage', 'variance']].to_string(index=False))


## Figure 3 — marker gene prediction quality  [MAIN/SUPP]

For each of three canonical CRC compartment markers (PIGR, COL1A1, IGKC), show:

- **Hexbin**: density of true vs predicted expression on the test set.
- **Within-quartile r table**: split true expression into 4 bins (Q1 = bottom 25%, Q4 = top 25%) and compute Pearson r within each bin.

The within-quartile r is a **cell-type-free proxy** for the question "is the high global r driven by binary detection (low vs high) or by genuine magnitude prediction?" If the model is doing pure binary detection:
- Q1 r and Q4 r will both be small (no within-class structure).
- The global r is essentially "are the high cells predicted higher than the low cells".

If the model is doing magnitude prediction:
- Q4 r in particular should be moderate-to-high (predictions track variation among the truly high-expressing cells).

A proper compartment-stratified version of this analysis is deferred until cell type annotation is available.

In [ ]:
# Three canonical compartment markers — kept for reuse in Figure 5.
MARKER_GENES = [
    ('PIGR',   'epithelial'),
    ('COL1A1', 'stromal'),
    ('IGKC',   'lymphoid'),
]
MARKER_GENES = [(g, c) for g, c in MARKER_GENES if g in df['gene_name'].values]

fig, axes = plt.subplots(1, len(MARKER_GENES), figsize=(5 * len(MARKER_GENES), 5),
                          sharey=False)
if len(MARKER_GENES) == 1: axes = [axes]

quartile_summary = []

for ax, (gene, target_comp) in zip(axes, MARKER_GENES):
    row = df[df['gene_name'] == gene].iloc[0]
    col = int(row['col_idx'])
    t = y_true[:, col]
    p = y_pred[:, col]
    r_global = float(row['test_pearson_s42'])

    # Hexbin (single colormap, log density)
    hb = ax.hexbin(t, p, gridsize=50, cmap='viridis', bins='log',
                    mincnt=1, rasterized=True)
    cb = plt.colorbar(hb, ax=ax, fraction=0.04, pad=0.03, shrink=0.8)
    cb.set_label('log count', fontsize=8)
    cb.ax.tick_params(labelsize=7)

    # diagonal
    lims = [min(t.min(), p.min()) - 0.1, max(t.max(), p.max()) + 0.1]
    ax.plot(lims, lims, 'r--', lw=0.9, alpha=0.7, label='y = x')

    # Within-quartile r
    q_edges = np.percentile(t, [0, 25, 50, 75, 100])
    q_labels = ['Q1 (low)', 'Q2', 'Q3', 'Q4 (high)']
    quartile_rs = []
    for qi in range(4):
        lo, hi = q_edges[qi], q_edges[qi + 1]
        if qi == 3:
            m = (t >= lo) & (t <= hi)
        else:
            m = (t >= lo) & (t < hi)
        if m.sum() >= 30:
            r_q, _ = stats.pearsonr(t[m], p[m])
            quartile_rs.append((q_labels[qi], r_q, m.sum()))
        else:
            quartile_rs.append((q_labels[qi], np.nan, m.sum()))

    quartile_summary.append((gene, r_global, quartile_rs))

    txt_lines = [f"global r = {r_global:.3f}", '─' * 14, 'within-quartile r:']
    for label, r_q, n_q in quartile_rs:
        if not np.isnan(r_q):
            txt_lines.append(f"  {label:10s} {r_q:+.3f}  (n={n_q:,})")
        else:
            txt_lines.append(f"  {label:10s}    n/a  (n={n_q})")

    ax.text(0.04, 0.97, '\n'.join(txt_lines),
            transform=ax.transAxes, va='top', ha='left',
            fontsize=7.5, family='monospace',
            bbox=dict(boxstyle='round,pad=0.4', facecolor='white',
                      edgecolor='#bbbbbb', alpha=0.95))

    ax.set_xlabel('true expression (z-score)')
    ax.set_ylabel('predicted expression (z-score)')
    comp_color = COMPARTMENT_COLORS.get(target_comp, '#444444')
    ax.set_title(f'{gene}  ({target_comp})', color=comp_color)
    ax.set_xlim(*lims); ax.set_ylim(*lims)
    ax.legend(loc='lower right', fontsize=7)

fig.suptitle('Figure 3 — marker gene prediction with within-quartile r breakdown',
             fontsize=11, y=1.02)

if SAVE_FIGS:
    fig.savefig(FIG_DIR / 'fig3_marker_quartile.pdf', bbox_inches='tight')
    fig.savefig(FIG_DIR / 'fig3_marker_quartile.png', bbox_inches='tight')
plt.show()

# Compact summary print
print('\nWithin-quartile r summary:')
print(f'{"gene":<8} {"global":>8} {"Q1":>8} {"Q2":>8} {"Q3":>8} {"Q4":>8}')
for gene, r_glob, qrs in quartile_summary:
    qvals = [f"{r:>+7.3f}" if not np.isnan(r) else "    n/a" for _, r, _ in qrs]
    print(f'{gene:<8} {r_glob:>+7.3f} ' + ' '.join(qvals))


## Figure 4 — model vs tile-mean baseline  [MAIN candidate or SUPP]

The strongest reviewer rebuttal: *"how do you know your r=0.32 isn't just predicting tile averages? It's a 4-tile test set."*

For each test cell, the tile-mean baseline predicts the gene expression as the *mean of all test cells in the same tile*. Pearson r between this and the true expression measures how much variance is explained purely by tile identity.

- Points above the y=x line → the model uses sub-tile (cell-level) information
- Points on the line → the model is essentially predicting tile means
- Points below → model is *worse* than tile mean (concerning for those genes)


In [ ]:
# Compute tile-mean baseline: for each test cell, predict the mean expression
# of all test cells in the same tile, per gene.
tile_ids = test_meta['tile_id'].values
unique_tiles_in_test = np.unique(tile_ids)

y_tile_mean = np.zeros_like(y_true)
for t in unique_tiles_in_test:
    m = (tile_ids == t)
    y_tile_mean[m, :] = y_true[m, :].mean(axis=0, keepdims=True)

# Per-gene r against tile-mean baseline
def per_gene_pearson(yt, yp):
    yt_c = yt - yt.mean(axis=0, keepdims=True)
    yp_c = yp - yp.mean(axis=0, keepdims=True)
    num = (yt_c * yp_c).sum(axis=0)
    den = np.sqrt((yt_c**2).sum(axis=0) * (yp_c**2).sum(axis=0)) + 1e-9
    return num / den

# baseline_r = how well does tile-mean explain true expression
baseline_r = per_gene_pearson(y_true, y_tile_mean)
df['baseline_tile_r'] = baseline_r[df['col_idx'].values]
df['model_minus_tile'] = df['test_pearson_s42'] - df['baseline_tile_r']

fig, axes = plt.subplots(1, 2, figsize=(11, 5), gridspec_kw={'wspace': 0.32})

# ---- panel a: scatter of model r vs tile-mean baseline r ----
ax = axes[0]
for comp, sub in df.groupby('compartment'):
    ax.scatter(sub['baseline_tile_r'], sub['test_pearson_s42'],
               c=COMPARTMENT_COLORS.get(comp, '#9aa5b1'),
               s=14, alpha=0.7, edgecolors='none',
               label=f'{comp} (n={len(sub)})')

lims = [-0.05, max(1.0, df[['baseline_tile_r', 'test_pearson_s42']].max().max() + 0.05)]
ax.plot(lims, lims, 'k--', lw=0.8, alpha=0.6, label='y = x')
ax.set_xlabel('tile-mean baseline r')
ax.set_ylabel('model r')
ax.set_title('a   model vs tile-mean baseline (per gene)')
ax.set_xlim(*lims); ax.set_ylim(*lims)

# Annotate top genes
for _, row in df.head(8).iterrows():
    ax.annotate(row['gene_name'],
                (row['baseline_tile_r'], row['test_pearson_s42']),
                xytext=(3, 3), textcoords='offset points',
                fontsize=7, color='#1b5e20', fontweight='bold')
ax.legend(loc='lower right', fontsize=6.5, markerscale=1.5, ncol=2)

# ---- panel b: histogram of model − baseline ----
ax = axes[1]
diffs = df['model_minus_tile']
ax.hist(diffs, bins=40, color='#4A90D9', edgecolor='white', alpha=0.85)
ax.axvline(0, color='red', ls='--', lw=1.0, label='no improvement')
ax.axvline(diffs.mean(), color='black', ls=':', lw=1.0,
           label=f'mean = {diffs.mean():.3f}')
ax.set_xlabel('model r  −  tile-mean baseline r')
ax.set_ylabel('# genes')
ax.set_title('b   improvement over tile-mean baseline')
n_above = (diffs > 0).sum()
n_above_05 = (diffs > 0.05).sum()
n_below = (diffs < 0).sum()
ax.text(0.97, 0.97,
        f"{n_above}/{len(diffs)} genes\nabove baseline\n\n"
        f"{n_above_05} genes\n>0.05 above\n\n"
        f"{n_below} genes\nbelow baseline",
        transform=ax.transAxes, va='top', ha='right',
        fontsize=8, family='monospace',
        bbox=dict(boxstyle='round,pad=0.4', facecolor='white',
                  edgecolor='#cccccc', alpha=0.95))
ax.legend(loc='upper left', fontsize=8)

if SAVE_FIGS:
    fig.savefig(FIG_DIR / 'fig4_tile_baseline.pdf', bbox_inches='tight')
    fig.savefig(FIG_DIR / 'fig4_tile_baseline.png', bbox_inches='tight')
plt.show()

print(f'\nMean model r           : {df["test_pearson_s42"].mean():.4f}')
print(f'Mean tile-baseline r   : {df["baseline_tile_r"].mean():.4f}')
print(f'Mean improvement       : {diffs.mean():.4f}')
print(f'Genes where model > baseline: {n_above}/{len(diffs)} ({n_above/len(diffs):.0%})')


## Figure 5 — spatial heatmap of marker genes  [MAIN — likely the hero figure]

For three compartment markers (PIGR, COL1A1, IGKC), compare predicted vs true expression in their **actual spatial location** within each test tile.

This is what reviewers and pathologists want to see. If the model is genuinely learning compartment-specific expression from H&E, the spatial pattern of (pred) should mirror (true) — distinct gland-like structures for PIGR, fibrous bands for COL1A1, scattered foci for IGKC.

Layout: 3 genes × 3 test tiles × {true, pred} = up to 18 panels. Each panel pair shares a colormap range so the comparison is fair.


In [ ]:
SPATIAL_GENES = [g for g, _ in MARKER_GENES]   # same 3 markers as Figure 3

# pick min(3, n_test_tiles) tiles
tiles_for_plot = sorted(test_meta['tile_id'].unique())[:3]
n_tiles = len(tiles_for_plot)
n_genes = len(SPATIAL_GENES)

# Layout: rows = genes, cols = (true, pred) × n_tiles
fig, axes = plt.subplots(n_genes, 2 * n_tiles,
                          figsize=(3.0 * n_tiles + 0.5, 3.0 * n_genes),
                          squeeze=False)

# Build a per-tile cell mask
tile_mask = {t: (test_meta['tile_id'].values == t) for t in tiles_for_plot}

for gi, gene in enumerate(SPATIAL_GENES):
    if gene not in df['gene_name'].values:
        continue
    row = df[df['gene_name'] == gene].iloc[0]
    col = int(row['col_idx'])
    t_vals = y_true[:, col]
    p_vals = y_pred[:, col]
    r_glob = float(row['test_pearson_s42'])

    # Robust shared color limits (1st–99th percentile)
    vmin, vmax = np.percentile(np.concatenate([t_vals, p_vals]), [1, 99])
    vabs = max(abs(vmin), abs(vmax))
    norm = TwoSlopeNorm(vmin=-vabs, vcenter=0, vmax=vabs)

    for ti, tile in enumerate(tiles_for_plot):
        m = tile_mask[tile]
        sub = test_meta[m]
        x = sub['x_centroid'].values
        y = sub['y_centroid'].values

        ax_t = axes[gi][2 * ti]
        ax_p = axes[gi][2 * ti + 1]

        ax_t.scatter(x, y, c=t_vals[m], cmap='RdBu_r', norm=norm,
                     s=2.0, alpha=0.85, edgecolors='none', rasterized=True)
        ax_p.scatter(x, y, c=p_vals[m], cmap='RdBu_r', norm=norm,
                     s=2.0, alpha=0.85, edgecolors='none', rasterized=True)

        # Per-tile r for this gene
        if m.sum() >= 5:
            r_tile, _ = stats.pearsonr(t_vals[m], p_vals[m])
        else:
            r_tile = np.nan

        if gi == 0:
            ax_t.set_title(f'tile {tile}\ntrue', fontsize=9)
            ax_p.set_title(f'tile {tile}\npred  (r={r_tile:.2f})', fontsize=9)
        else:
            ax_t.set_title(f'true', fontsize=9)
            ax_p.set_title(f'pred  (r={r_tile:.2f})', fontsize=9)

        for ax_ in [ax_t, ax_p]:
            ax_.set_aspect('equal', adjustable='datalim')
            ax_.set_xticks([]); ax_.set_yticks([])
            for s in ax_.spines.values(): s.set_visible(False)

    # Row label on the leftmost panel
    axes[gi][0].set_ylabel(f'{gene}\n(global r={r_glob:.2f})',
                            fontsize=10, fontweight='bold',
                            rotation=0, ha='right', va='center', labelpad=20)

    # Add a colorbar on the far right of this row
    cbar_ax = fig.add_axes([0.92, 0.92 - (gi + 1) * (0.85 / n_genes) + 0.02,
                             0.015, (0.85 / n_genes) * 0.6])
    cb = mpl.colorbar.ColorbarBase(cbar_ax,
                                    cmap=plt.get_cmap('RdBu_r'),
                                    norm=norm, orientation='vertical')
    cb.set_label('z-score', fontsize=8)
    cb.ax.tick_params(labelsize=7)

fig.subplots_adjust(left=0.08, right=0.90, hspace=0.35, wspace=0.10)
fig.suptitle(f'Figure 5 — spatial expression: true vs predicted (CRC, test tiles)',
             fontsize=11, y=0.99)

if SAVE_FIGS:
    fig.savefig(FIG_DIR / 'fig5_spatial_heatmap.pdf', bbox_inches='tight')
    fig.savefig(FIG_DIR / 'fig5_spatial_heatmap.png', bbox_inches='tight')
plt.show()


## Figure 6 — per-tile decomposition  [SUPP]

Sanity check: which test tile is contributing how much to the per-gene Pearson r? If one tile is dominating (or one tile is silently failing), this heatmap will show it.

Rows = top 30 genes by overall r. Columns = test tiles. Cell color = per-tile Pearson r.


In [ ]:
def per_gene_per_tile_r(y_true, y_pred, tile_ids, gene_cols):
    """Return DataFrame (genes × tiles) of per-tile Pearson r."""
    out = {}
    for t in np.unique(tile_ids):
        m = (tile_ids == t)
        if m.sum() < 5:
            continue
        rs = per_gene_pearson(y_true[m][:, gene_cols], y_pred[m][:, gene_cols])
        out[int(t)] = rs
    return pd.DataFrame(out)

top_for_heatmap = df.head(30).copy()
gene_cols = top_for_heatmap['col_idx'].values.astype(int)

per_tile_r = per_gene_per_tile_r(y_true, y_pred, tile_ids, gene_cols)
per_tile_r.index = top_for_heatmap['gene_name'].values    # gene names as rows
per_tile_r['overall'] = top_for_heatmap['test_pearson_s42'].values

# Reorder cols: tiles first, then overall
cols_order = [c for c in per_tile_r.columns if c != 'overall'] + ['overall']
per_tile_r = per_tile_r[cols_order]

fig, ax = plt.subplots(figsize=(0.6 * len(cols_order) + 2.5, 0.28 * len(per_tile_r) + 1.2))
sns.heatmap(per_tile_r, annot=True, fmt='.2f',
            cmap='RdYlGn', vmin=-0.1, vmax=1.0, center=0.3,
            linewidths=0.4, linecolor='white',
            cbar_kws={'label': 'Pearson r', 'shrink': 0.6},
            ax=ax,
            annot_kws={'fontsize': 6.5})

# Color compartment annotations on row labels
for i, gene in enumerate(per_tile_r.index):
    comp = df[df['gene_name'] == gene]['compartment'].iloc[0]
    ax.get_yticklabels()[i].set_color(COMPARTMENT_COLORS.get(comp, '#444444'))

# Highlight 'overall' column
ax.axvline(len(cols_order) - 1, color='black', lw=1.2)
ax.set_xlabel('test tile')
ax.set_ylabel('')
ax.set_title('per-tile Pearson r (top 30 genes by overall r)', loc='left')

if SAVE_FIGS:
    fig.savefig(FIG_DIR / 'fig6_per_tile_r.pdf', bbox_inches='tight')
    fig.savefig(FIG_DIR / 'fig6_per_tile_r.png', bbox_inches='tight')
plt.show()

# Quick numerical summary
print('\nPer-tile mean r (top 30 genes only):')
print(per_tile_r.iloc[:, :-1].mean().to_string())


## Figure 7 — variance & mean calibration  [SUPP]

Already computed in the cross-dataset diagnostic notebook, but included here for self-containment. CRC's slope on the variance plot was +0.71 (moderate shrinkage, healthy direction); we report the same metric here as a within-paper sanity check.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5), gridspec_kw={'wspace': 0.30})

for ax, (xkey, ykey, title) in zip(axes, [
    ('true_std_z',  'pred_std_z',  'a   variance calibration'),
    ('true_mean_z', 'pred_mean_z', 'b   mean calibration'),
]):
    sc = ax.scatter(df[xkey], df[ykey], c=df['test_pearson_s42'],
                    cmap='RdYlGn', vmin=-0.05, vmax=0.8,
                    s=15 + (df['coverage'] - df['coverage'].min()) /
                            (df['coverage'].max() - df['coverage'].min() + 1e-9) * 90,
                    alpha=0.7, edgecolor='black', lw=0.2, rasterized=True)

    coef = np.polyfit(df[xkey], df[ykey], 1)
    if 'std' in xkey:
        lim_max = max(df[xkey].max(), df[ykey].max()) * 1.05
        lims = (0, lim_max)
    else:
        xy_min = min(df[xkey].min(), df[ykey].min())
        xy_max = max(df[xkey].max(), df[ykey].max())
        pad = 0.05 * (xy_max - xy_min + 1e-9)
        lims = (xy_min - pad, xy_max + pad)
        ax.axhline(0, color='#888', lw=0.5); ax.axvline(0, color='#888', lw=0.5)

    ax.plot(lims, lims, 'k--', lw=0.8, label='y = x')
    xs = np.array(lims)
    ax.plot(xs, coef[0] * xs + coef[1], color='#1f77b4', lw=1.2,
            label=f'fit: y={coef[0]:.2f}x+{coef[1]:.2f}')

    r_calib, _ = stats.pearsonr(df[xkey], df[ykey])
    ax.text(0.04, 0.97, f'r = {r_calib:+.3f}\nslope = {coef[0]:+.3f}',
            transform=ax.transAxes, va='top', fontsize=8, family='monospace',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                      edgecolor='#cccccc', alpha=0.95))

    ax.set_xlabel(f'{xkey.replace("_z", "")}  (TRUE)')
    ax.set_ylabel(f'{ykey.replace("_z", "")}  (PRED)')
    ax.set_title(title)
    ax.set_xlim(*lims); ax.set_ylim(*lims)
    ax.legend(loc='lower right', fontsize=7.5)

cb = fig.colorbar(sc, ax=axes, fraction=0.025, pad=0.02, shrink=0.8)
cb.set_label('test Pearson r')

if SAVE_FIGS:
    fig.savefig(FIG_DIR / 'fig7_calibration.pdf', bbox_inches='tight')
    fig.savefig(FIG_DIR / 'fig7_calibration.png', bbox_inches='tight')
plt.show()


## Summary — what's in each figure

| Figure | Content | Use |
|---|---|---|
| 1 | Per-gene r distribution + top-N curves + r vs coverage / variance | **MAIN**: shape of result |
| 2 | Top-30 genes ranked + compartment composition by r quartile | **MAIN**: biological interpretation |
| 3 | Marker-gene hexbin + within-quartile r (cell-type-free magnitude diagnostic) | MAIN/SUPP |
| 4 | Model r vs tile-mean baseline | **MAIN/SUPP**: addresses tile-leakage concern |
| 5 | Spatial heatmap, true vs predicted, 3 markers × 3 tiles | **MAIN (hero)**: visual evidence |
| 6 | Per-tile per-gene r heatmap (top 30 genes) | SUPP |
| 7 | Variance & mean calibration | SUPP |

### Deferred (need cell type annotation)
- Compartment-stratified scatter for marker genes (the "compartment vs magnitude" diagnostic). Once you have cell type from Xenium / MoE predictions, this is one cell of code: filter cells by `cell_type == 'epithelial'` and recompute r within that subset for PIGR.

### Deferred (need frozen baseline)
- Per-gene scatter of frozen-baseline r vs LoRA r, to quantify "what does LoRA add". When ready, drop the prediction npy at `FROZEN_DIR` (set in cell 1) and add a panel to Figure 4.

### My recommended Figure 2 in the paper

If forced to pick **one** of these as the centerpiece, it's **Figure 5** — the spatial heatmap. Reviewers and pathologists immediately understand "the prediction recovers the histological pattern".

Figures 2 and 4 are the supporting cast — Figure 2 quantifies *which* genes work, Figure 4 is the rebuttal-readiness card for tile-leakage concerns.

Figure 3's within-quartile r is a partial substitute for the compartment diagnostic. If Q4 r ≪ global r for a gene like PIGR, it suggests the high global r is binary-detection-driven and the magnitude framing should be questioned.